In [0]:
# ── Cell 1: Import all libraries ────────────────────
from datetime import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar
import pandas as pd

print("✓ Libraries loaded!")
print(f"✓ Pipeline year: {datetime.now().year}")


✓ Libraries loaded!
✓ Pipeline year: 2026


In [0]:
# ── Cell 2: Dynamic dates — no hardcoding! ──────────
current_year = datetime.now().year

# Start = January 1st of current year
sdt = datetime(current_year, 1, 1)

# End = December 31st of current year
edt = datetime(current_year, 12, 31)

print(f"Start date: {sdt.date()}")
print(f"End date:   {edt.date()}")
print(f"Year:       {current_year}")

Start date: 2026-01-01
End date:   2026-12-31
Year:       2026


In [0]:
# ── Cell 3: Get US Federal Holidays ─────────────────
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=sdt, end=edt)

print(f"Total US Federal Holidays in {current_year}: {len(holidays)}")
print("\nAll holidays:")
for h in holidays:
    print(f"  {h.date()} — {h.day_name()}")

Total US Federal Holidays in 2026: 11

All holidays:
  2026-01-01 — Thursday
  2026-01-19 — Monday
  2026-02-16 — Monday
  2026-05-25 — Monday
  2026-06-19 — Friday
  2026-07-03 — Friday
  2026-09-07 — Monday
  2026-10-12 — Monday
  2026-11-11 — Wednesday
  2026-11-26 — Thursday
  2026-12-25 — Friday


In [0]:
# ── Cell 4: Create full date DataFrame ──────────────
df_hol = pd.DataFrame()
df_hol['date']           = pd.date_range(start=sdt, end=edt)
df_hol['year']           = df_hol['date'].dt.year
df_hol['month']          = df_hol['date'].dt.month
df_hol['day']            = df_hol['date'].dt.day
df_hol['day_name']       = df_hol['date'].dt.day_name()
df_hol['week_number']    = df_hol['date'].dt.isocalendar().week
df_hol['is_holiday']     = [1 if v in holidays else 0 for v in df_hol['date']]
df_hol['is_weekend']     = df_hol['date'].dt.dayofweek.isin([5,6]).astype(int)
df_hol['is_working_day'] = ((df_hol['is_holiday']==0) & (df_hol['is_weekend']==0)).astype(int)

print(f"Total days: {len(df_hol)}")
print(f"Total holidays: {df_hol['is_holiday'].sum()}")
print(f"Total weekends: {df_hol['is_weekend'].sum()}")
print(f"Total working days: {df_hol['is_working_day'].sum()}")
df_hol.head(10)

Total days: 365
Total holidays: 11
Total weekends: 104
Total working days: 250


,date,year,month,day,day_name,week_number,is_holiday,is_weekend,is_working_day
0,2026-01-01,2026,1,1,Thursday,1,1,0,0
1,2026-01-02,2026,1,2,Friday,1,0,0,1
2,2026-01-03,2026,1,3,Saturday,1,0,1,0
3,2026-01-04,2026,1,4,Sunday,1,0,1,0
4,2026-01-05,2026,1,5,Monday,2,0,0,1
5,2026-01-06,2026,1,6,Tuesday,2,0,0,1
6,2026-01-07,2026,1,7,Wednesday,2,0,0,1
7,2026-01-08,2026,1,8,Thursday,2,0,0,1
8,2026-01-09,2026,1,9,Friday,2,0,0,1
9,2026-01-10,2026,1,10,Saturday,2,0,1,0


In [0]:
%skip
# ── Cell 5: Save to Delta table in Unity Catalog ────
df_spark = spark.createDataFrame(df_hol)

df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.us_holiday_calendar")

print("Holiday calendar saved to Delta table!")
print(f"Table: workspace.default.us_holiday_calendar")
print(f"Total rows saved: {df_spark.count()}")

In [0]:
from datetime import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar
import pandas as pd

current_year = datetime.now().year
sdt = datetime(current_year, 1, 1)
edt = datetime(current_year, 12, 31)

cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=sdt, end=edt)

df_hol = pd.DataFrame()
df_hol['date']           = pd.date_range(start=sdt, end=edt)
df_hol['year']           = df_hol['date'].dt.year.astype(int)
df_hol['month']          = df_hol['date'].dt.month.astype(int)
df_hol['day']            = df_hol['date'].dt.day.astype(int)
df_hol['day_name']       = df_hol['date'].dt.day_name()
df_hol['week_number']    = df_hol['date'].dt.isocalendar().week.astype(int)
df_hol['is_holiday']     = [1 if v in holidays else 0 for v in df_hol['date']]
df_hol['is_weekend']     = df_hol['date'].dt.dayofweek.isin([5,6]).astype(int)
df_hol['is_working_day'] = ((df_hol['is_holiday']==0) & (df_hol['is_weekend']==0)).astype(int)

# Convert date column to string to avoid type issues
df_hol['date'] = df_hol['date'].astype(str)

print(f"Rows ready: {len(df_hol)}")

# Save using SQL instead
df_spark = spark.createDataFrame(df_hol)
df_spark.createOrReplaceTempView("temp_holiday")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.us_holiday_calendar
    AS SELECT * FROM temp_holiday
""")

print("✓ Table created successfully!")

# Verify
count = spark.sql("SELECT COUNT(*) as total FROM workspace.default.us_holiday_calendar").collect()[0][0]
print(f"✓ Total rows in table: {count}")

Rows ready: 365
✓ Table created successfully!
✓ Total rows in table: 365


In [0]:
%skip
print(spark.sql("SELECT current_catalog()").collect()[0][0])
print(spark.sq03:55 PM (17m)
7

print(spark.sql("SELECT current_catalog()").collect()[0][0])print(spark.sql("SELECT current_database()").collect()[0][0])

(2) Spark Jobs
Job 0 
View
(Stages: 1/1)
Job 1 
View
(Stages: 1/1)
 
workspace
default$0l("SELECT current_database()").collect()[0][0])

In [0]:
# ── See only holidays ────────────────────────────────
spark.sql("""
    SELECT date, day_name, is_holiday, is_weekend, is_working_day
    FROM workspace.default.us_holiday_calendar
    WHERE is_holiday = 1
    ORDER BY date
""").show(truncate=False)

+----------+---------+----------+----------+--------------+
|date      |day_name |is_holiday|is_weekend|is_working_day|
+----------+---------+----------+----------+--------------+
|2026-01-01|Thursday |1         |0         |0             |
|2026-01-19|Monday   |1         |0         |0             |
|2026-02-16|Monday   |1         |0         |0             |
|2026-05-25|Monday   |1         |0         |0             |
|2026-06-19|Friday   |1         |0         |0             |
|2026-07-03|Friday   |1         |0         |0             |
|2026-09-07|Monday   |1         |0         |0             |
|2026-10-12|Monday   |1         |0         |0             |
|2026-11-11|Wednesday|1         |0         |0             |
|2026-11-26|Thursday |1         |0         |0             |
|2026-12-25|Friday   |1         |0         |0             |
+----------+---------+----------+----------+--------------+



In [0]:
# ── Load ALL historical years 2020 to 2026 ──────────
from datetime import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar
import pandas as pd

cal = USFederalHolidayCalendar()

historical_start = 2020
current_year = datetime.now().year

all_years_df = []

for year in range(historical_start, current_year + 1):
    sdt = datetime(year, 1, 1)
    edt = datetime(year, 12, 31)
    holidays = cal.holidays(start=sdt, end=edt)

    df_year = pd.DataFrame()
    df_year['date']           = pd.date_range(start=sdt, end=edt)
    df_year['year']           = df_year['date'].dt.year.astype(int)
    df_year['month']          = df_year['date'].dt.month.astype(int)
    df_year['day']            = df_year['date'].dt.day.astype(int)
    df_year['day_name']       = df_year['date'].dt.day_name()
    df_year['week_number']    = df_year['date'].dt.isocalendar().week.astype(int)
    df_year['is_holiday']     = [1 if v in holidays else 0 for v in df_year['date']]
    df_year['is_weekend']     = df_year['date'].dt.dayofweek.isin([5,6]).astype(int)
    df_year['is_working_day'] = ((df_year['is_holiday']==0) & (df_year['is_weekend']==0)).astype(int)
    df_year['date']           = df_year['date'].astype(str)

    all_years_df.append(df_year)
    print(f"✓ {year} ready — {len(df_year)} days")

# Combine all years
df_all = pd.concat(all_years_df, ignore_index=True)
print(f"\n✓ Total rows all years: {len(df_all)}")

# Save to same managed table
df_spark = spark.createDataFrame(df_all)
df_spark.createOrReplaceTempView("temp_all_years")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.us_holiday_calendar
    AS SELECT * FROM temp_all_years
""")

print("\n✓ All years saved to us_holiday_calendar!")

# Verify — show summary
spark.sql("""
    SELECT year,
           COUNT(*) as total_days,
           SUM(is_holiday) as holidays,
           SUM(is_weekend) as weekends,
           SUM(is_working_day) as working_days
    FROM workspace.default.us_holiday_calendar
    GROUP BY year
    ORDER BY year
""").show()

✓ 2020 ready — 366 days
✓ 2021 ready — 365 days
✓ 2022 ready — 365 days
✓ 2023 ready — 365 days
✓ 2024 ready — 366 days
✓ 2025 ready — 365 days
✓ 2026 ready — 365 days

✓ Total rows all years: 2557

✓ All years saved to us_holiday_calendar!
+----+----------+--------+--------+------------+
|year|total_days|holidays|weekends|working_days|
+----+----------+--------+--------+------------+
|2020|       366|      10|     104|         252|
|2021|       365|      12|     104|         249|
|2022|       365|      10|     105|         250|
|2023|       365|      11|     105|         249|
|2024|       366|      11|     104|         251|
|2025|       365|      11|     104|         250|
|2026|       365|      11|     104|         250|
+----+----------+--------+--------+------------+

